In [12]:
Report_Type="S"


StatementMeta(, 91a80668-2837-4c63-a9ef-dded8de61e92, 15, Finished, Available, Finished, False)

In [13]:
import pandas as pd

# Define the data
data = [('Excel Generation','D'), ('RestAPI_Export','S')]

# Create DataFrame
df = pd.DataFrame(data, columns=['Notebook','Type'])
#df=spark.createDataFrame(df)
display(df)
# 4. Extract value to a variable before dropping column
# .iloc[0] gets the first value from the 'Notebook' column
#notebook_name = df['Notebook'].iloc[0] if not df.empty else None
#notebook_ = df.filter((df["Type"] == Report_Type))#.select("Notebook")
notebook = df[df['Type'] == Report_Type][['Notebook']]
# 5. Drop the 'Type' column
df = df.drop(columns=['Type'])
notebook_name = notebook['Notebook'].iloc[0]
display(df)
print(notebook_name)


StatementMeta(, 91a80668-2837-4c63-a9ef-dded8de61e92, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b9e88ed4-54a7-407c-9e2b-1e5e18e8ef0f)

SynapseWidget(Synapse.DataFrame, e4053516-d4bf-4778-914b-003a889bb2fe)

RestAPI_Export


## Load Report List Table

In [24]:
from pyspark.sql.functions import when, col


df = spark.sql("SELECT * FROM ITS_Home.silver.ITS_Home_Report_List WHERE Report = 'ITS Home - FACETS DF Home Input Detail Control'")
###Blocked Temp by Amit
# df = df.withColumn(
#     "Report Type",
#     when(col("Fabric") == 1, "D").otherwise("S")
# )
# display(Report_Type)
# df=df.filter(df["Report Type"]==Report_Type)
#Report=df.filter(df["Run"]==1)

display(df)
#Report = Report.drop("Run","Fabric","Report Type","filename")
###Blocked Temp by Amit
Report = df
display(Report)

StatementMeta(, 91a80668-2837-4c63-a9ef-dded8de61e92, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ad95c934-06dd-4645-8835-7e0482418494)

SynapseWidget(Synapse.DataFrame, c8f40994-d0a0-42cd-929e-7876c05380ad)

## Convert Table to Dictionary

In [25]:

# Convert each Row object to a dictionary
first_ten_rows = Report.head(100)
list_of_dictionaries = [row.asDict() for row in first_ten_rows]


# Print the resulting list of dictionaries
print(list_of_dictionaries)

StatementMeta(, 91a80668-2837-4c63-a9ef-dded8de61e92, 29, Finished, Available, Finished, False)

[{'Report': 'ITS Home - FACETS DF Home Input Detail Control', 'Jurisdiction': 'DC', 'filename': 'Home_DF_CNTL_DETAIL_DC080_212183_213336_', 'Fabric': 1, 'Business Category': 'Home', 'Run': 0}, {'Report': 'ITS Home - FACETS DF Home Input Detail Control', 'Jurisdiction': 'MD', 'filename': 'Home_DF_CNTL_DETAIL_MD190_212183_213336_', 'Fabric': 1, 'Business Category': 'Home', 'Run': 0}, {'Report': 'ITS Home - FACETS DF Home Input Detail Control', 'Jurisdiction': 'MEDICAID', 'filename': 'Home_DF_CNTL_DETAIL_MDCAID_212183_213336_', 'Fabric': 1, 'Business Category': 'GP', 'Run': 0}, {'Report': 'ITS Home - FACETS DF Home Input Detail Control', 'Jurisdiction': 'DSNP', 'filename': 'Home_DF_CNTL_DETAIL_DSNP_212183_213336_', 'Fabric': 1, 'Business Category': 'GP', 'Run': 0}, {'Report': 'ITS Home - FACETS DF Home Input Detail Control', 'Jurisdiction': 'MEDICARE_ADVANTAGE', 'filename': 'Home_DF_CNTL_DETAIL_MAPPO_212183_213336_', 'Fabric': 1, 'Business Category': 'GP', 'Run': 0}, {'Report': 'ITS Home 

## Call "Excel Generation" notebook from here

In [26]:
from notebookutils import mssparkutils

# Ensure this variable is defined as a list of dictionaries
reports_to_export = list_of_dictionaries


child_notebook_path = notebook_name

for item in reports_to_export:
    # Prepare the dictionary for arguments
    notebook_args = {
        "Report_Name": item['Report'],
        "Jurisdiction": item['Jurisdiction'],
        "Business_Category": item['Business Category']
    }    
    
    # Use 'arguments' keyword and a safe timeout (e.g., 1800 seconds / 30 mins)
    run_result = mssparkutils.notebook.run(
        child_notebook_path,
        timeout_seconds=43200,
        arguments=notebook_args
    )
    
    print(f"Notebook '{child_notebook_path}' run complete with result: {run_result}")


StatementMeta(, 91a80668-2837-4c63-a9ef-dded8de61e92, 31, Finished, Available, Finished, False)

Py4JJavaError: An error occurred while calling o52468.throwExceptionIfHave.
: com.microsoft.spark.notebook.msutils.NotebookExecutionException: name 'File_location' is not defined
---------------------------------------------------------------------------NameError                                 Traceback (most recent call last)Cell In[11], line 1
----> 1 file_response = requests.get(File_location, headers=headers)
      2 if file_response.status_code == 200:
      3     file_content = file_response.content
NameError: name 'File_location' is not definedYou can check driver log or snapshot for detailed error info! See how to check logs: https://go.microsoft.com/fwlink/?linkid=2157243 .
	at com.microsoft.spark.notebook.workflow.JobSessionClient.runCell(JobSessionClient.scala:419)
	at com.microsoft.spark.notebook.workflow.JobSessionClient.$anonfun$run$4(JobSessionClient.scala:191)
	at com.microsoft.spark.notebook.workflow.JobSessionClient.$anonfun$run$4$adapted(JobSessionClient.scala:174)
